# PhoBERT -- Combined augmentation (BT + EDA + LLM + VOZ)

## Dependencies

In [ ]:
!pip install -q transformers datasets huggingface_hub scikit-learn accelerate sentencepiece py_vncorenlp
!pip install -q -U datasets

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 10.5 MB/s eta 0:00:00


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: Enable GPU in Runtime > Change runtime type.")

Device: cuda
GPU: Tesla T4


In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami, HfApi

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)
user_info = whoami()
print(f"Authenticated as: {user_info['name']}")

Authenticated as: AnoraLee


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Hate_Speech_Detection")
DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
AUGMENTED_DIR = DATA_DIR / "augmented"
MODELS_DIR = PROJECT_DIR / "models"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_DIR}")

Mounted at /content/drive
Project root: /content/drive/MyDrive/Hate_Speech_Detection


In [ ]:
MODEL_NAME = "vinai/phobert-base"
MAX_LENGTH = 128
SEED = 42
LABELS = ["CLEAN", "OFFENSIVE", "HATE"]

label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for label, index in label2id.items()}

## Chuẩn hoá & Phân từ dùng chung

In [ ]:
import os
import re
import unicodedata
import pandas as pd
import py_vncorenlp

def text_key(text):
    """Normalize a text for exact-duplicate / leakage matching."""
    text = unicodedata.normalize("NFKC", str(text)).strip().lower()
    return re.sub(r"\s+", " ", text)

def add_key(df, source_col="text_raw"):
    df = df.copy()
    df["_key"] = df[source_col].map(text_key)
    return df

_vncorenlp_dir = "/content/vncorenlp"
if not os.path.exists(_vncorenlp_dir):
    os.makedirs(_vncorenlp_dir, exist_ok=True)
    py_vncorenlp.download_model(save_dir=_vncorenlp_dir)
rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=_vncorenlp_dir)

def segment_text(text):
    try:
        sentences = rdrsegmenter.word_segment(str(text))
        return " ".join(sentences)
    except Exception:
        return str(text)

def apply_segmentation(df, source_col="text_raw", target_col="text"):
    df = df.copy()
    df[target_col] = df[source_col].map(segment_text)
    return df

In [ ]:
TOXIC_TEENCODE_MAP = {
    r"\bko\b": "không", r"\bhok\b": "không", r"\bdc\b": "được", r"\bđc\b": "được",
    r"\bj\b": "gì", r"\bbt\b": "bình thường", r"\btrc\b": "trước", r"\bnhg\b": "nhưng",
    r"\bthg\b": "thằng",
    r"\bdm\b": "địt mẹ", r"\bđm\b": "địt mẹ", r"\bdkm\b": "địt con mẹ", r"\bđkm\b": "địt con mẹ",
    r"\bvkl\b": "vãi lồn", r"\bvcl\b": "vãi lồn", r"\bvl\b": "vãi lồn", r"\bkl\b": "cái lồn",
    r"\bcc\b": "cục cứt", r"\bcđm\b": "cộng đồng mạng", r"\bml\b": "mặt lồn",
    r"\bđjt\b": "địt", r"\bdjt\b": "địt", r"\bdit\b": "địt",
    r"\bloz\b": "lồn", r"\blon\b": "lồn",
    r"\bcac\b": "cặc", r"\bcặk\b": "cặc", r"\bđb\b": "đầu buồi",
    r"\bcút\b": "cút", r"\bđĩ\b": "đĩ", r"\bphò\b": "phò"
}

def normalize_teencode(text):
    for pattern, replacement in TOXIC_TEENCODE_MAP.items():
        text = re.sub(pattern, replacement, str(text), flags=re.IGNORECASE)
    return text

def apply_teencode_normalization(df, col="text_raw"):
    df = df.copy()
    df[col] = df[col].apply(normalize_teencode)
    return df

## Load dev/test đã đóng băng (Giai đoạn 1)

In [ ]:
dev_raw = pd.read_csv(PROCESSED_DIR / "dev.csv")
test_raw = pd.read_csv(PROCESSED_DIR / "test.csv")

dev_raw = dev_raw.rename(columns={"text": "text_raw"}) if "text_raw" not in dev_raw.columns else dev_raw
test_raw = test_raw.rename(columns={"text": "text_raw"}) if "text_raw" not in test_raw.columns else test_raw

dev_df = apply_teencode_normalization(dev_raw.dropna(subset=["text_raw"]).reset_index(drop=True))
test_df = apply_teencode_normalization(test_raw.dropna(subset=["text_raw"]).reset_index(drop=True))

dev_keys = set(dev_df["text_raw"].map(text_key))
test_keys = set(test_df["text_raw"].map(text_key))

dev_df = apply_segmentation(dev_df)
test_df = apply_segmentation(test_df)

print(f"Validation: {dev_df.shape}")
print(f"Test: {test_df.shape}")

Validation: (2650, 5)
Test: (6576, 5)


## Load 5 nguồn: baseline + BT + EDA + LLM + VOZ

In [ ]:
source_files = {
    "baseline": AUGMENTED_DIR / "aug_baseline.csv",
    "BT": AUGMENTED_DIR / "aug_bt.csv",
    "EDA": AUGMENTED_DIR / "aug_eda.csv",
    "LLM": AUGMENTED_DIR / "aug_llm.csv",
}
for name, file_path in source_files.items():
    assert file_path.exists(), f"Missing: {file_path}"
    df = pd.read_csv(file_path)
    print(f"{name}: {len(df):,} dòng -- {df['label'].value_counts().to_dict()}")

baseline: 85,127 dòng -- {'CLEAN': 60034, 'OFFENSIVE': 19470, 'HATE': 5623}
BT: 4,234 dòng -- {'OFFENSIVE': 3407, 'HATE': 827}
EDA: 134,412 dòng -- {'CLEAN': 60034, 'OFFENSIVE': 57575, 'HATE': 16803}
LLM: 88,191 dòng -- {'CLEAN': 60034, 'OFFENSIVE': 20504, 'HATE': 7653}
VOZ: 803 dòng -- {'HATE': 550, 'OFFENSIVE': 253}


## Gộp + loại nhãn xung đột + loại leakage với dev/test (1 lần, trên toàn bộ 5 nguồn)

In [ ]:
dfs = []
for name, path in source_files.items():
    df = apply_teencode_normalization(pd.read_csv(path))
    df["augmentation_set"] = name
    dfs.append(df)

combined_raw = pd.concat(dfs, ignore_index=True)
combined_raw = add_key(combined_raw)
print(f"Tổng thô (trước dedup): {len(combined_raw):,}")
print(combined_raw["label"].value_counts().to_dict())

conflict_keys = set(
    combined_raw.groupby("_key")["label"].nunique().loc[lambda c: c > 1].index
)
blocked_keys = conflict_keys | dev_keys | test_keys

combined_df = (
    combined_raw[~combined_raw["_key"].isin(blocked_keys)]
    .drop_duplicates("_key")
    .drop(columns="_key")
    .reset_index(drop=True)
)
print(f"\nTrain cuối cùng (đã dedup + loại xung đột/leakage): {len(combined_df):,}")
print(combined_df["label"].value_counts().to_dict())
print(f"Loại vì nhãn xung đột: {len(conflict_keys)}")

Tổng thô (trước dedup): 312,767
{'CLEAN': 180102, 'OFFENSIVE': 101209, 'HATE': 31456}

Train cuối cùng (đã dedup + loại xung đột/leakage): 141,503
{'OFFENSIVE': 62112, 'CLEAN': 59266, 'HATE': 20125}
Loại vì nhãn xung đột: 57


## Sanity check -- không còn leakage giữa các split

In [ ]:
train_keys = set(combined_df["text_raw"].map(text_key))
print(f"Train ∩ Validation: {len(train_keys & dev_keys)}")
print(f"Train ∩ Test: {len(train_keys & test_keys)}")
assert len(train_keys & dev_keys) == 0 and len(train_keys & test_keys) == 0, "Vẫn còn leakage!"

Train ∩ Validation: 0
Train ∩ Test: 0


## Tokenizer + hàm dùng chung

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def prepare_split(df):
    output = df[["text", "label"]].dropna().copy()
    assert output["label"].isin(LABELS).all(), "Unexpected label found."
    output["label"] = output["label"].map(label2id)
    return Dataset.from_pandas(output, preserve_index=False)

def build_tokenized_dataset(train_df, dev_df, test_df):
    dataset = DatasetDict({
        "train": prepare_split(train_df),
        "validation": prepare_split(dev_df),
        "test": prepare_split(test_df),
    })
    tokenized = dataset.map(tokenize_function, batched=True)
    tokenized = tokenized.rename_column("label", "labels")
    model_columns = [
        c for c in ["input_ids", "attention_mask", "token_type_ids", "labels"]
        if c in tokenized["train"].column_names
    ]
    tokenized.set_format("torch", columns=model_columns)
    print(tokenized)
    return tokenized

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

In [ ]:
combined_df = apply_segmentation(combined_df)
tokenized = build_tokenized_dataset(combined_df, dev_df, test_df)

Map:   0%|          | 0/141503 [00:00<?, ? examples/s]

Map:   0%|          | 0/2650 [00:00<?, ? examples/s]

Map:   0%|          | 0/6576 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 141503
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 2650
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 6576
    })
})


## Metrics

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    """Unified metric schema -- MUST stay identical across all 3 notebooks
    so results in the report are directly comparable."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "weighted_f1": f1_score(labels, predictions, average="weighted"),
        "hate_f1": f1_score(labels, predictions, labels=[LABELS.index("HATE")], average="macro"),
    }

## Train

In [ ]:
import json
from transformers import TrainingArguments, AutoModelForSequenceClassification, Trainer, set_seed

set_seed(SEED)

def train_experiment(experiment_name, model_dir_name, tokenized, push_to_hub_repo=None):
    """Load a FRESH PhoBERT model and train one experiment end to end.
    Always instantiates a new model -- never reuses a model/trainer object
    from a previous cell, to avoid accidental weight leakage between runs."""
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=3, label2id=label2id, id2label=id2label,
    )
    model_dir = MODELS_DIR / model_dir_name

    training_args = TrainingArguments(
        output_dir=str(model_dir / "_checkpoints"),
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    test_metrics = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
    print(f"[{experiment_name}] test metrics: {test_metrics}")

    trainer.save_model(model_dir)
    tokenizer.save_pretrained(model_dir)

    experiment_results_dir = RESULTS_DIR / experiment_name
    experiment_results_dir.mkdir(parents=True, exist_ok=True)

    metrics_path = experiment_results_dir / f"metrics_{experiment_name}.json"
    with open(metrics_path, "w") as f:
        json.dump(test_metrics, f, indent=2)

    print(f"Model saved to: {model_dir}")
    print(f"Metrics saved to: {metrics_path}")

    if push_to_hub_repo:
        api = HfApi()
        api.create_repo(repo_id=push_to_hub_repo, repo_type="model", private=True, exist_ok=True)
        trainer.model.push_to_hub(push_to_hub_repo, private=True,
                                  commit_message=f"Upload PhoBERT {experiment_name} model")
        tokenizer.push_to_hub(push_to_hub_repo, private=True,
                              commit_message="Upload PhoBERT tokenizer")
        api.upload_file(
            path_or_fileobj=str(metrics_path),
            path_in_repo="test_metrics.json",
            repo_id=push_to_hub_repo, repo_type="model",
        )
        print(f"Uploaded: https://huggingface.co/{push_to_hub_repo}")

    return trainer, test_metrics

In [ ]:
trainer, test_metrics = train_experiment(
    experiment_name="combined",
    model_dir_name="combined_phobert",
    tokenized=tokenized,
    push_to_hub_repo=f"{whoami()['name']}/vietnamese-hsd-phobert-combined",
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  543MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...h6ljtmj/model.safetensors:   0%|          |  548kB /  540MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded: https://huggingface.co/AnoraLee/vietnamese-hsd-phobert-combined
